In [227]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [259]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_tumbling"
#
tumbling_int = ts_step_int * 100
allowed_lateness_intervals = 1
allowed_lateness_ms = allowed_lateness_intervals * tumbling_int
#
order_tn = (
    Tn.source(order_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // tumbling_int) * tumbling_int + tumbling_int + allowed_lateness_ms)
    .distinct()
)
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: (
            (x["ts"] // tumbling_int) * tumbling_int + tumbling_int,
            x["customer_id"]
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": by[0] - tumbling_int,
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={
            "orders": 0,
            "total_price": 0,
            "last_ts": 0
        }
    )
)
#
def get_expired_tumbling_windows(max_ts):
    current_closed_end = (max_ts // tumbling_int) * tumbling_int
    return [
        {"window_end": current_closed_end - i * tumbling_int}
        for i in range(allowed_lateness_intervals + 1)
        if current_closed_end - i * tumbling_int >= tumbling_int
    ]

window_tn = (
    group_by_tn
    .join_equi(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x)
        .flatmap(lambda max_ts: get_expired_tumbling_windows(max_ts)),
        lambda l: l["window_end"],
        lambda r: r["window_end"],
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))



In [260]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# tumbling_int=100

print("=== Window [0,100) running ===")
push(1, 100, 10)  # window_end=100
push(1, 200, 50)  # window_end=100, same bucket
push(2, 150, 70)  # window_end=100, customer 2

print("\n=== Window [0,100) closing → Emit ===")
push(1, 300, 105) # watermark=105 > 100 → both customers fire!
                  # customer 1: orders=2, total=300
                  # customer 2: orders=1, total=150

print("\n=== Window [100,200) running ===")
push(2, 200, 120) # window_end=200
push(1, 100, 150) # window_end=200
push(2, 300, 180) # window_end=200

print("\n=== Late Arrival for window [0,100) – still inside allowed lateness ===")
push(1, 500, 90)  # ts=90 → window_end=100, watermark=180 < 200 → correction!
                  # customer 1: +{orders=3, total=800} / -{orders=2, total=300}

print("\n=== Window [100,200) closing → Emit ===")
push(1, 100, 205) # watermark=205 > 200 → both customers fire!
                  # customer 1: orders=2, total=400
                  # customer 2: orders=2, total=500

=== Window [0,100) running ===

=== Window [0,100) closing → Emit ===
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50}, 1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 2, 'orders': 1, 'total_price': 150, 'last_ts': 70}, 1)

=== Window [100,200) running ===

=== Late Arrival for window [0,100) – still inside allowed lateness ===
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50}, -1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 3, 'total_price': 800, 'last_ts': 90}, 1)

=== Window [100,200) closing → Emit ===
expel: ({'window_start': 100, 'window_end': 200, 'customer_id': 2, 'orders': 2, 'total_price': 500, 'last_ts': 180}, 1)
expel: ({'window_start': 100, 'window_end': 200, 'customer_id': 1, 'orders': 2, 'total_price': 400, 'last_ts': 150}, 1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 2, 'orders'

In [303]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
tumbling_int = ts_step_int * 100
hop_int = tumbling_int // 2  # hop=50, window=100 → jedes Event in 2 Buckets
# Wie viele Hops dürfen Events maximal zu spät ankommen?
allowed_lateness_hops = 3
# Das entspricht in Sekunden/Millisekunden:
allowed_lateness_ms = allowed_lateness_hops * hop_int  # 2 * 50 = 100
#
order_tn = (
    Tn.source(order_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // tumbling_int) * tumbling_int + (tumbling_int * 2) + allowed_lateness_ms)
    .distinct()
)
#
def assign(ts):
    first_end = (ts // hop_int) * hop_int + hop_int
    x = [
        first_end + i * hop_int 
        for i in range(tumbling_int // hop_int)
        if first_end + i * hop_int >= tumbling_int
    ]
    print(f"assign: ts: {ts}, windows: {x}")
    return x
#
group_by_tn = (
    order_tn
    .flatmap(lambda x: [
        {**x, "ts_orig": x["ts"], "window_end_hint": we}
        for we in assign(x["ts"])
    ])
    .group_by_agg(
        by_function=lambda x: (x["window_end_hint"], x["customer_id"]),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": by[0] - tumbling_int,
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts_orig"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0}
    )
)
#
def triggers(max_ts):
    current_window_end = (max_ts // hop_int) * hop_int
    x = [
        current_window_end - i * hop_int
        for i in range(allowed_lateness_hops + 1)
        if current_window_end - i * hop_int >= tumbling_int
    ]
    print(f"triggers: current_window_end: {current_window_end}, windows: {x}")
    return x
#
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x)
        .flatmap(lambda max_ts: [{"window_end": y} for y in triggers(max_ts)]),
        lambda l, r: l["window_end"] == r["window_end"],
        lambda l, _: l
    )
)._peek("emit")
#
built_tn = Tn.build(window_tn.sink(sink_str))

In [304]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# tumbling=100, hop=50 → windows: [0,100), [50,150), [100,200), ...
# ts=10  → window_ends: 100 (only [0,100))
# ts=60  → window_ends: 100, 150 (in [0,100) AND [50,150)!)
# ts=105 → window_ends: 150, 200 (in [50,150) AND [100,200)!)

print("=== Events in window [0,100) ===")
push(1, 100, 10)  # → only window_end=100
push(1, 200, 60)  # → window_end=100 AND 150

print("\n=== [0,100) closes → Emit ===")
push(1, 300, 105) # watermark=105 >     100
                  # → [0,100): orders=2, total=300 (ts=10 + ts=60)
                  # ts=105 goes into [50,150) AND [100,200)

print("\n=== [50,150) closes → Emit ===")
push(1, 400, 155) # watermark=155 > 150
                  # → [50,150): orders=2, total=500 (ts=60 + ts=105)

print("\n=== [100,200) closes → Emit ===")
push(1, 100, 205) # watermark=205 > 200
                  # → [100,200): orders=2, total=700 (ts=105 + ts=155)

print("\n=== late arrivals ===")
push(1, 42, 49)
push(1, 23, 101)

print("\n=== [200,300) closes → Emit ===")
push(1, 100, 305) # watermark=305 > 300
                  # → [200,300): orders=2, total=700 (ts=105 + ts=155)

print("\n=== late arrival ===")
push(1, 4711, 101)

print("\n=== too late arrival ===")
push(1, 4711, 49)



=== Events in window [0,100) ===
assign: ts: 10, windows: [100]
triggers: current_window_end: 0, windows: []
assign: ts: 60, windows: [100, 150]
triggers: current_window_end: 50, windows: []
triggers: current_window_end: 0, windows: []

=== [0,100) closes → Emit ===
assign: ts: 105, windows: [150, 200]
triggers: current_window_end: 100, windows: [100]
triggers: current_window_end: 50, windows: []
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}, 1)

=== [50,150) closes → Emit ===
assign: ts: 155, windows: [200, 250]
triggers: current_window_end: 150, windows: [150, 100]
triggers: current_window_end: 100, windows: [100]
emit: ({'window_start': 50, 'window_end': 150, 'customer_id': 1, 'orders': 2, 'total_price': 500, 'last_ts': 105}, 1)

=== [100,200) closes → Emit ===
assign: ts: 205, windows: [250, 300]
triggers: current_window_end: 200, windows: [200, 150, 100]
triggers: current_window_end: 150, windows: [150, 100]
emit: (

In [300]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    res = built_tn.latest()
    if res:
        print(f"Output bei ts={ts}: {res}")

# tumbling=100, hop=50 → windows: [0,100), [50,150), [100,200), ...

print("=== Normaler Anlauf ===")
push(1, 100, 10)   # Fenster-Ende: 100
push(1, 200, 60)   # Fenster-Ende: 100, 150

print("\n=== [0,100) schließt ===")
push(1, 300, 105)  # watermark=105 -> schließt [0,100)

print("\n=== DER ZEITSPRUNG (Simuliert Lücke / Batch-Schub) ===")
# Dieses Event zieht max_ts schlagartig auf 500 hoch.
# triggers(500) liefert nur noch Fenster ab 350+ (500 - 3 * 50).
# Das Fenster für 450/500 wird geöffnet.
push(1, 100, 500)  

print("\n=== ERLAUBTE LATE ARRIVAL (Sollte durchgehen) ===")
# ts=410 ist vollkommen valide (Latenz erlaubt bis zu ~150ms Verspätung)
# assign(410) packt es korrekt in die Fenster-Enden 450 und 500.
# ABER: Da max_ts bei 500 bleibt, feuert der Max-Stream kein Update.
# Da das Fenster 450 nicht mehr in triggers(500) aktiv im Join-Zustand gehalten wird,
# verschwindet dieses Update im Nichts!
push(1, 50, 410)  

print("\n=== TOO LATE ARRIVAL (Sollte gefiltert werden) ===")
# ts=100 ist viel zu spät bei max_ts=500 -> fliegt flach durch .expire() raus.
push(1, 999, 100)

=== Normaler Anlauf ===
assign: ts: 10, windows: [100]
triggers: current_window_end: 0, windows: []
assign: ts: 60, windows: [100, 150]
triggers: current_window_end: 50, windows: []
triggers: current_window_end: 0, windows: []

=== [0,100) schließt ===
assign: ts: 105, windows: [150, 200]
triggers: current_window_end: 100, windows: [100]
triggers: current_window_end: 50, windows: []
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}, 1)
Output bei ts=105: {'orders_hopping': [{'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}]}

=== DER ZEITSPRUNG (Simuliert Lücke / Batch-Schub) ===
assign: ts: 500, windows: [550, 600]
assign: ts: 105, windows: [150, 200]
assign: ts: 60, windows: [100, 150]
assign: ts: 10, windows: [100]
triggers: current_window_end: 500, windows: [500, 450, 400, 350]
triggers: current_window_end: 100, windows: [100]
emit: ({'window_start': 0, 'window_end':

In [251]:
tumbling_int = ts_step_int * 100
hop_int = tumbling_int // 2
allowed_lateness_hops = 3
allowed_lateness_ms = allowed_lateness_hops * hop_int
#
order_tn = (
    Tn.source(order_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // tumbling_int) * tumbling_int + (tumbling_int * 2) + allowed_lateness_ms)  # Bug gefixt
    .distinct()
)
#
def cumulative_window_ends(ts, tumbling_int, hop_int):
    # Periode ist FIX (tumbling_int-Raster), nur das Ende wächst
    period_start = (ts // tumbling_int) * tumbling_int
    relative_ts = ts - period_start
    k = tumbling_int // hop_int
    return [period_start + m * hop_int
            for m in range(1, k + 1)
            if m * hop_int > relative_ts]
#
group_by_tn = (
    order_tn
    .flatmap(lambda x: [
        {**x, "ts_orig": x["ts"], "window_end_hint": we}
        for we in cumulative_window_ends(x["ts"], tumbling_int, hop_int)
    ])
    .group_by_agg(
        by_function=lambda x: (x["window_end_hint"], x["customer_id"]),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": ((by[0] - 1) // tumbling_int) * tumbling_int,  # ⚠️ geändert: fixer Periodenstart statt by[0]-tumbling_int
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts_orig"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0}
    )
)
#
def get_expired_windows_with_lateness(max_ts):
    current_window_end = (max_ts // hop_int) * hop_int
    return [
        {"window_end": current_window_end - i * hop_int}
        for i in range(allowed_lateness_hops + 1)
        if current_window_end - i * hop_int >= 0
    ]
#
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: x)
        .flatmap(lambda max_ts: get_expired_windows_with_lateness(max_ts)),
        lambda l, r: l["window_end"] == r["window_end"],
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))

In [252]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# tumbling=100, step=50 → windows immer ab gleichem start:
# ts=10  → cumulate_window_ends: [50, 100]
# ts=60  → cumulate_window_ends: [100]
# ts=110 → cumulate_window_ends: [150, 200]
# ts=160 → cumulate_window_ends: [200]

print("=== Events im Zyklus [0,100) ===")
push(1, 100, 10)  # → window_ends: 50, 100

print("\n=== [0,50) schliesst + neues Event → Emit ===")
push(1, 200, 60)  # watermark → 75 > 50 → [0,50) feuert: orders=1, total=100
                  # ts=60 → window_ends: 100

print("\n=== [0,100) schliesst + neues Event → Emit ===")
push(1, 300, 110) # watermark → 125 > 100 → [0,100) feuert: orders=2, total=300 (kumulativ!)
                  # ts=110 → window_ends: 150, 200

print("\n=== [100,150) schliesst + neues Event → Emit ===")
push(1, 400, 160) # watermark → 175 > 150 → [100,150) feuert: orders=1, total=300
                  # ts=160 → window_ends: 200

print("\n=== [100,200) schliesst + neues Event → Emit ===")
push(1, 500, 210) # watermark → 225 > 200 → [100,200) feuert: orders=2, total=700 (kumulativ!)

=== Events im Zyklus [0,100) ===

=== [0,50) schliesst + neues Event → Emit ===
expel: ({'window_start': 0, 'window_end': 50, 'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10}, 1)

=== [0,100) schliesst + neues Event → Emit ===
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}, 1)

=== [100,150) schliesst + neues Event → Emit ===
expel: ({'window_start': 100, 'window_end': 150, 'customer_id': 1, 'orders': 1, 'total_price': 300, 'last_ts': 110}, 1)

=== [100,200) schliesst + neues Event → Emit ===
expel: ({'window_start': 100, 'window_end': 200, 'customer_id': 1, 'orders': 2, 'total_price': 700, 'last_ts': 160}, 1)


In [ ]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# max_size=100, step=25 → windows [0,25), [0,50), [0,75), [0,100) für jeden Customer

print("=== Erst-Event, geht in alle 4 Buckets ===")
push(1, 100, 10)  # → window_ends: 25, 50, 75, 100

print("\n=== Step 25 überschritten → [0,25) emittet ===")
push(1, 200, 30)  # watermark=30 > 25 → [0,25) feuert mit orders=1, total=100

print("\n=== Step 50 überschritten → [0,50) emittet ===")
push(1, 300, 55)  # watermark=55 > 50 → [0,50) feuert mit orders=2, total=300

print("\n=== Step 75 überschritten → [0,75) emittet ===")
push(1, 400, 80)  # watermark=80 > 75 → [0,75) feuert mit orders=3, total=600

print("\n=== max_size überschritten → [0,100) emittet, neuer Zyklus ===")
push(1, 500, 105) # watermark=105 > 100 → [0,100) feuert mit orders=4, total=1000
                  # ts=105 startet neuen Zyklus: window_ends 125, 150, 175, 200

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

import functools

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_session"
#
gap_int = ts_step_int * 20
max_session_int = gap_int * 10
#
order_tn = (
    Tn.source(order_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // max_session_int) * max_session_int + max_session_int)
    .distinct()
)
#
def assign(ts):
    first_end = (ts // gap_int) * gap_int + gap_int
    x = [
        first_end + i * gap_int
        for i in range(max_session_int // gap_int)
        if first_end + i * gap_int >= max_session_int
    ]
    print(f"assign: ts: {ts}, windows: {x}")
    return x
#
session_tn = order_tn.flatmap(lambda x: [
        {**x, "ts_orig": x["ts"], "window_start": w["start"], "window_end_hint": w["end"]}
        for w in assign(x["ts"])
    ])

group_by_tn = (
    session_tn
    .group_by_agg(
        by_function=lambda x: (x["window_start"], x["window_end_hint"], x["customer_id"]),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": by[0],
            "window_end": by[1],
            "customer_id": by[2],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts_orig"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0}
    )
)
#
def trigger_session(max_ts):
    return [{"trigger_ts": max_ts - gap_int}]

window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: x)
        .flatmap(trigger_session),
        lambda l, r: l["window_end"] == r["trigger_ts"],
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [317]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# gap=20, max_session=200

print("=== Session customer 1 läuft ===")
push(1, 100, 0)   # session_bucket=0
push(1, 200, 10)  # session_bucket=0, verlängert
push(1, 300, 15)  # session_bucket=0, verlängert

print("\n=== Session customer 2 läuft parallel ===")
push(2, 150, 5)   # session_bucket=0
push(2, 250, 18)  # session_bucket=0, verlängert

print("\n=== Gap customer 1 abgelaufen → Emit ===")
push(2, 100, 60)  # watermark=60 > 15+20=35 → customer 1 feuert! orders=3, total=600

print("\n=== Gap customer 2 abgelaufen → Emit ===")
push(1, 100, 90)  # watermark=90 > 18+20=38 → customer 2 feuert! orders=2, total=400

print("\n=== Hard Reset bei max_session_int → neue session_bucket ===")
push(1, 500, 205) # session_bucket=200 → frischer state!
push(1, 500, 210) # session_bucket=200, verlängert

print("\n=== Gap neue Session customer 1 abgelaufen → Emit ===")
push(2, 50, 250)  # watermark=250 > 210+20=230 → customer 1 feuert! orders=2, total=1000

=== Session customer 1 läuft ===
assign: ts: 0, windows: [4000]


TypeError: 'int' object is not subscriptable

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_session_threshold"
#
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
#
order_tn = (
    Tn.source(order_source_str)
    .expire(lambda x: x["value"]["ts"],
            lambda x: (x // max_session_int) * max_session_int + max_session_int)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .distinct()
)
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: (
            x["customer_id"],
            (x["ts"] // max_session_int) * max_session_int
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "customer_id": by[0],
            "session_bucket": by[1],
            "window_end": agg["last_ts"] + gap_int,
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0}
    )
)._peek("group")
#
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: {"ts": x}),
        lambda l, r: (r["ts"] > l["last_ts"] + gap_int
                      or l["total_price"] > 1000
                      or l["orders"] > 20),
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))



In [ ]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

print("=== Session läuft, kein Emit ===")
push(1, 100, 0)   # customer 1, total=100, session_bucket=0
push(1, 200, 5)   # customer 1, total=300
push(1, 300, 10)  # customer 1, total=600

print("\n=== Threshold >1000 → Emit, aber Session läuft weiter ===")
push(1, 500, 15)  # customer 1, total=1100 → feuert! session_bucket=0 bleibt offen

print("\n=== Session läuft weiter im selben bucket ===")
push(1, 100, 20)  # customer 1, total=1200, last_ts=20

print("\n=== Gap abgelaufen → Emit mit kumulativem total ===")
push(2, 50, 60)   # watermark=60 > last_ts=20 + gap=20 → customer 1 feuert mit total=1200!

print("\n=== customer 2 threshold ===")
push(2, 400, 65)
push(2, 400, 70)
push(2, 400, 75)  # customer 2, total=1250 → feuert!

print("\n=== Hard Reset bei max_session_int ===")
push(1, 999, max_session_int)  # customer 1 neue session_bucket → frischer state

In [ ]:
import cloudpickle

built_tn.reset()
order_generator = OrderGenerator()

state_size_kb_float_list = []
for i in range(100):
    print(f"Step {i + 1}")
    order_message_dict_list = [order_generator.generate() for _ in range(100)]
    built_tn.push(order_source_str, order_message_dict_list)
    built_tn.latest()
    state_size_kb_float = len(cloudpickle.dumps(built_tn)) / 1024
    state_size_kb_float_list.append(state_size_kb_float)
#
print(f"Last 10 state sizes (KB): {state_size_kb_float_list[-10:]}")
